In [1]:
%pip install -q google-genai pydantic python-dotenv

In [2]:
from pydantic import BaseModel, Field
from typing import List
from google import genai
from google.genai import types
from dotenv import load_dotenv
import os


In [3]:
load_dotenv()
APIKEY = os.environ.get("GEMINI_API_KEY")

In [17]:
class Actor(BaseModel):
    nome: str = Field(description="O nome dos atores e atrizes")
    personagem: str = Field(description="O nome do personagem que interpretaram.")


class MovieSummary(BaseModel):
    title: str = Field(description="O título do filme.")
    release_year: int = Field(description="O ano em que o filme foi lançado.")
    genres: List[str] = Field(description="Uma lista de gêneros para o filme.")
    main_cast: List[Actor] = Field(description="Uma lista dos 3 principais atores.")
    budget: int = Field(description="O orçamento do filme em dólares.")
    revenue: int = Field(description="A receita do filme em dólares.")

In [18]:
client = genai.Client(api_key=APIKEY)

config = types.GenerateContentConfig(
    temperature=0,
    response_mime_type="application/json",
    response_schema=MovieSummary,  # Seu modelo Pydantic entra direto aqui
    system_instruction="Voce é um assistente de cinema ajudante. Sempre envolva sua resposta final estritamente em JSON correspondente ao esquema solicitado." # System prompt simplificado
)



In [19]:
movie_name = "Matrix" 

response = client.models.generate_content(
    model="gemini-2.5-flash-lite", # gemini-2.5-flash é o recomendado para uso geral rápido
    contents=f"Extraia as informações sobre o filme: {movie_name}",
    config=config
)


In [20]:
filme_estruturado = response.parsed
filme_estruturado

MovieSummary(title='Matrix', release_year=1999, genres=['Action', 'Sci-Fi'], main_cast=[Actor(nome='Keanu Reeves', personagem='Neo'), Actor(nome='Laurence Fishburne', personagem='Morpheus'), Actor(nome='Carrie-Anne Moss', personagem='Trinity')], budget=63000000, revenue=465000000)

In [21]:
def get_movie_summary(movie_name: str) -> MovieSummary:
    response = client.models.generate_content(
        model="gemini-2.5-flash-lite",
        contents=f"Extraia as informações sobre o filme: {movie_name}",
        config=config
    )
    return response.parsed


In [22]:
print(get_movie_summary("Titanic"))

title='Titanic' release_year=1997 genres=['Romance', 'Drama'] main_cast=[Actor(nome='Leonardo DiCaprio', personagem='Jack Dawson'), Actor(nome='Kate Winslet', personagem='Rose DeWitt Bukater'), Actor(nome='Billy Zane', personagem='Cal Hockley')] budget=200000000 revenue=2201647264


In [24]:
print(get_movie_summary("Matrix 2"))

title='Matrix Reloaded' release_year=2003 genres=['Action', 'Sci-Fi'] main_cast=[Actor(nome='Keanu Reeves', personagem='Neo'), Actor(nome='Laurence Fishburne', personagem='Morpheus'), Actor(nome='Carrie-Anne Moss', personagem='Trinity')] budget=150000000 revenue=739400000


In [23]:
print(get_movie_summary("Jurassic Park"))

title='Jurassic Park' release_year=1993 genres=['Adventure', 'Sci-Fi', 'Thriller'] main_cast=[Actor(nome='Sam Neill', personagem='Dr. Alan Grant'), Actor(nome='Laura Dern', personagem='Dr. Ellie Sattler'), Actor(nome='Jeff Goldblum', personagem='Dr. Ian Malcolm')] budget=63000000 revenue=914000000
